# 批量处理 MTSD 图片

使用 `slicer.py` 中的 `process_image` 方法批量处理所有图片，并保存到目标路径。

## 数据路径
- 输入：`/Users/weixianfu/Documents/Datas/mtsd` (train_full, train_partial, val)
- 输出：`/Users/weixianfu/Documents/Datas/mtsd-resized` (保持相同文件夹结构)

In [1]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

sys.path.append(str(Path.cwd().parent))
from src.io import process_image

In [2]:
INPUT_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd")
OUTPUT_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd-resized")
DATASET_TYPES = ["train_full", "train_partial", "val"]
MAX_WORKERS = 8

In [3]:
def get_image_names(dataset_type: str) -> list:
    images_dir = INPUT_ROOT / dataset_type / "images"
    image_names = []
    for img_path in images_dir.glob("*.jpg"):
        image_names.append(img_path.stem)
    return image_names

def process_single_image(name: str, dataset_type: str):
    input_path = INPUT_ROOT / dataset_type
    output_path = OUTPUT_ROOT / dataset_type
    process_image(name=name, input_path=input_path, output_path=output_path)
    return name

In [4]:
for dataset_type in DATASET_TYPES:
    print(f"\n处理 {dataset_type}...")
    image_names = get_image_names(dataset_type)
    print(f"找到 {len(image_names)} 张图片")
    
    if len(image_names) == 0:
        print(f"跳过 {dataset_type}，没有找到图片")
        continue
    
    output_path = OUTPUT_ROOT / dataset_type
    (output_path / "images").mkdir(parents=True, exist_ok=True)
    (output_path / "labels").mkdir(parents=True, exist_ok=True)
    print(f"输出目录已准备: {output_path}")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_single_image, name, dataset_type): name 
            for name in image_names
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"处理 {dataset_type}"):
            name = futures[future]
            future.result()
    
    print(f"{dataset_type} 处理完成")


处理 train_full...
找到 36589 张图片
输出目录已准备: /Users/weixianfu/Documents/Datas/mtsd-resized/train_full


处理 train_full: 100%|██████████| 36589/36589 [03:24<00:00, 179.20it/s]


train_full 处理完成

处理 train_partial...
找到 53377 张图片
输出目录已准备: /Users/weixianfu/Documents/Datas/mtsd-resized/train_partial


处理 train_partial: 100%|██████████| 53377/53377 [04:54<00:00, 181.08it/s]


train_partial 处理完成

处理 val...
找到 5320 张图片
输出目录已准备: /Users/weixianfu/Documents/Datas/mtsd-resized/val


处理 val: 100%|██████████| 5320/5320 [00:18<00:00, 290.45it/s]

val 处理完成


In [4]:
from src.io.slicer import process_val_image

# 仅针对 val 数据集，使用 process_val_image 重新处理
val_dataset_type = "val"
print(f"\n重新处理 {val_dataset_type} (val) ...")
image_names = get_image_names(val_dataset_type)
print(f"找到 {len(image_names)} 张图片")

if len(image_names) == 0:
    print(f"跳过 {val_dataset_type}，没有找到图片")
else:
    output_path = OUTPUT_ROOT / val_dataset_type
    (output_path / "images").mkdir(parents=True, exist_ok=True)
    (output_path / "labels").mkdir(parents=True, exist_ok=True)
    print(f"输出目录已准备: {output_path}")

    def process_single_val_image(name: str):
        input_path = INPUT_ROOT / val_dataset_type
        output_path = OUTPUT_ROOT / val_dataset_type
        process_val_image(name=name, input_path=input_path, output_path=output_path)
        return name

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_single_val_image, name): name
            for name in image_names
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc=f"处理 {val_dataset_type} (val)"):
            name = futures[future]
            future.result()

    print(f"{val_dataset_type} (val) 处理完成")




重新处理 val (val) ...
找到 5320 张图片
输出目录已准备: /Users/weixianfu/Documents/Datas/mtsd-resized/val


处理 val (val): 100%|██████████| 5320/5320 [00:48<00:00, 109.96it/s]

val (val) 处理完成
